# Fase 3: Feature Engineering y Preparación de Datos

## Objetivos de esta fase:
1. Clonar el repositorio y cargar el dataset limpio (`dataset_cleaned.csv`) generado en la fase anterior.
2. Aislar correctamente la variable objetivo (`Malware`) de las variables predictoras (features).
3. Realizar la división estratificada del dataset (*Train/Test Split*) para garantizar una representación equilibrada de las clases en los conjuntos de entrenamiento y test.
4. Evaluar y aplicar las transformaciones y el escalado de variables numéricas para optimizar el rendimiento de los modelos.

## 1 Clonar el repositorio y cargar el dataset limpio (`dataset_cleaned.csv`) generado en la fase anterior.

Automatizamos la carga del entorno clonando el repositorio y consumiendo directamente el dataset optimizado en la fase anterior, asegurando la trazabilidad completa del flujo de trabajo de Machine Learning.

In [15]:
# 1. Clonar el repositorio oficial desde GitHub para asegurar la persistencia en Colab
!git clone https://github.com/EliasHernandezBar/mini-antivirus-ai

# 2. Cambiar al directorio de trabajo del proyecto
import os
os.chdir('mini-antivirus-ai')

# 3. Cargar el dataset limpio exportado en la Fase 2
import pandas as pd
import numpy as np

df = pd.read_csv('data/dataset_cleaned.csv')

print("Dataset limpio cargado con éxito para la Fase 3[cite: 1].")
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.")

Cloning into 'mini-antivirus-ai'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 31 (delta 10), reused 25 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 2.05 MiB | 8.72 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Dataset limpio cargado con éxito para la Fase 3[cite: 1].
Dimensiones del dataset: 19611 filas y 70 columnas.


## 2 Aislar correctamente la variable objetivo (*Malware*) de las variables predictoras (*features*).

En supervised learning, es una buena práctica separar claramente la matriz de características (X) del vector objetivo (y) antes de aplicar cualquier transformación o división de datos.

In [16]:
# Definir la variable objetivo y las variables predictoras
target_col = 'Malware'

if target_col in df.columns:
    X = df.drop(columns=[target_col])
    y = df[target_col]

    print(f"Características predictoras (X): {X.shape[1]} columnas.")
    print(f"Variable objetivo (y): '{target_col}' con distribución:")
    print(y.value_counts(normalize=True) * 100)
else:
    print(f"La columna objetivo '{target_col}' no se encuentra en el DataFrame.")

Características predictoras (X): 69 columnas.
Variable objetivo (y): 'Malware' con distribución:
Malware
1    74.442915
0    25.557085
Name: proportion, dtype: float64


## 3 Realizar la división estratificada del dataset (*Train/Test Split*) para garantizar una representación equilibrada de las clases en los conjuntos de entrenamiento y test.

Ante datasets con desequilibrio de clases, utilizar stratify=y es una buena práctica obligatoria. Esto garantiza que tanto el conjunto de entrenamiento como el de test conserven exactamente la misma proporción de archivos benignos y maliciosos.

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el dataset en entrenamiento (80%) y prueba (20%) de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Dimensiones del conjunto de entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones del conjunto de prueba (X_test): {X_test.shape}")
print("\nDistribución de clases en el conjunto de entrenamiento (%):")
print(y_train.value_counts(normalize=True) * 100)

Dimensiones del conjunto de entrenamiento (X_train): (15688, 69)
Dimensiones del conjunto de prueba (X_test): (3923, 69)

Distribución de clases en el conjunto de entrenamiento (%):
Malware
1    74.445436
0    25.554564
Name: proportion, dtype: float64


## 4 Evaluar y aplicar las transformaciones y el escalado de variables numéricas para optimizar el rendimiento de los modelos.

Para evitar que variables con rangos numéricos muy amplios dominen sobre otras en ciertos algoritmos de Machine Learning, estandarizamos las características. Regla de oro de Machine Learning: El escalador (StandardScaler) se ajusta (fit) únicamente con el conjunto de entrenamiento y luego se aplica (transform) a ambos conjuntos, previniendo así cualquier fuga de información (data leakage).

In [18]:
from sklearn.preprocessing import StandardScaler

# Inicializar el objeto StandardScaler
scaler = StandardScaler()

# Ajustar únicamente con X_train y transformar X_train y X_test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir los resultados de nuevo a DataFrames para preservar los nombres de las columnas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("¡Escalado de variables completado con éxito!")
print(f"Media de la primera variable escalada en train (debe ser aprox. 0): {X_train_scaled.iloc[:, 0].mean():.5f}")
print(f"Desviación estándar de la primera variable escalada en train (debe ser 1): {X_train_scaled.iloc[:, 0].std():.5f}")

¡Escalado de variables completado con éxito!
Media de la primera variable escalada en train (debe ser aprox. 0): 0.00000
Desviación estándar de la primera variable escalada en train (debe ser 1): 1.00003


5: Persistencia de los conjuntos preparados

Exportamos los conjuntos de datos ya transformados y divididos a una subcarpeta estructurada (data/fase_3/) y los empaquetamos en formato comprimido. De esta forma, la siguiente fase consumirá directamente los splits listos para el modelado.

In [19]:
import os
import shutil

# 1. Crear la subcarpeta específica para la Fase 3 dentro de data/
os.makedirs('data/fase_3', exist_ok=True)

# 2. Guardar los archivos dentro de la nueva subcarpeta
X_train_scaled.to_csv('data/fase_3/X_train.csv', index=False)
X_test_scaled.to_csv('data/fase_3/X_test.csv', index=False)
y_train.to_csv('data/fase_3/y_train.csv', index=False)
y_test.to_csv('data/fase_3/y_test.csv', index=False)

# 3. Comprimir la carpeta en un archivo .zip para descargarla de golpe
shutil.make_archive('data_fase_3', 'zip', 'data/fase_3')

# 4. Forzar la descarga automática del zip a tu equipo local
from google.colab import files
files.download('data_fase_3.zip')

print("¡Archivos guardados en 'data/fase_3/' y archivo ZIP listo para descargar!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

¡Archivos guardados en 'data/fase_3/' y archivo ZIP listo para descargar!


## Conclusiones y Próximos Pasos

### Resumen de la Fase de Feature Engineering:
* **Separación de datos:** Se aislaron correctamente las 69 características predictoras (`X`) de la variable objetivo (`Malware`), confirmando el desequilibrio de clases (~74.4% malware, ~25.6% benignos).
* **División estratificada (*Train/Test Split*):** Se realizó un reparto 80/20 garantizando mediante estratificación (`stratify=y`) que la proporción de clases se mantuviera idéntica tanto en entrenamiento como en test.
* **Escalado seguro sin fugas de datos:** Se aplicó `StandardScaler` ajustando exclusivamente los parámetros sobre el conjunto de entrenamiento (`X_train`) y transformando posteriormente ambos conjuntos de forma independiente.
* **Persistencia:** Los datasets transformados (`X_train`, `X_test`, `y_train`, `y_test`) han sido exportados con éxito a la carpeta `data/` para su consumo directo en la fase de modelado.

### Decisiones pendientes para la Fase 4 (Modelado y Entrenamiento):
1. Seleccionar los algoritmos de clasificación base (ej. Árboles de Decisión, Random Forest) adecuados para este problema de ciberseguridad.
2. Definir las métricas de evaluación clave (Accuracy, Precision, Recall, F1-Score y Curva ROC-AUC) teniendo en cuenta el desequilibrio de clases.
3. Entrenar los modelos iniciales y evaluar su rendimiento sobre el conjunto de prueba (`X_test`).